# Phase 1 · Architecture 4 — LegalBlock-GQA 262M

In [3]:
%pip -q install "tokenizers>=0.20,<0.23" "transformers>=4.45,<5" "huggingface_hub>=0.26,<1" ijson cloudpickle bert-score rouge-score sacrebleu

Note: you may need to restart the kernel to use updated packages.


In [5]:
import os, sys, json, math, time, random, hashlib, socket, subprocess, cloudpickle, gc, zipfile
from pathlib import Path
from contextlib import nullcontext
from datetime import timedelta
import numpy as np
import torch, shutil
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from huggingface_hub import HfFileSystem, hf_hub_download
from kaggle_secrets import UserSecretsClient
import ijson

SESSION_START = time.monotonic()
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ['RAYON_NUM_THREADS'] = '4'
os.environ['OMP_NUM_THREADS'] = '2'
OUT = Path('/kaggle/working/legalblock_gqa')
OUT.mkdir(parents=True, exist_ok=True)
DATA = OUT / 'data'
DATA.mkdir(exist_ok=True)

source = Path("/kaggle/input/datasets/kumaralpha1/p1-v4-s1/legalblock_gqa")
shutil.copytree(source, OUT, dirs_exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
SOURCES = ['acts', 'high_courts', 'supreme_court', 'tribunals']
MIX = [.10, .40, .35, .15]
CFG = dict(vocab=32768, dim=960, layers=24, heads=15, kv_heads=3, ff=2560,
           window=256, global_every=4, seq=1024, long_seq=1536, micro=2, accum=8,
           lr=3e-4, min_lr=3e-5, warmup=150, wd=.1, seed=781,
           train_tokens=300000000, train_seconds=6.5*3600, eval_every=250,
           val_batches=24, mix=MIX, sources=SOURCES, out=str(OUT), data=str(DATA))
RESUME = '/kaggle/input/datasets/kumaralpha1/p1-v4-s1/legalblock_gqa/last.pt'
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])

## 1. Corpus and BPE

In [6]:
def documents(source):
    fs = HfFileSystem(token=os.environ['HF_TOKEN'])
    path = f'datasets/{REPO}/Phase_1_Pretraining/Train/train_{source}.json'
    with fs.open(path, 'rb', block_size=8 << 20) as f:
        first = f.read(4096).lstrip()[:1]
        f.seek(0)
        rows = ijson.items(f, 'item') if first == b'[' else (v for _, v in ijson.kvitems(f, ''))
        for row in rows:
            text = row if isinstance(row, (str, list)) else row['text']
            if isinstance(text, list):
                text = ' '.join(text)
            text = ' '.join(text.split())
            if text:
                key = int.from_bytes(hashlib.blake2b(text.encode(), digest_size=8).digest(), 'little')
                yield text, key % 1000 < 10, key

def bpe_texts():
    for source in SOURCES:
        size = 0
        for text, heldout, _ in documents(source):
            if not heldout:
                yield text
                size += len(text)
                if size >= 20000000:
                    break

tok = Tokenizer(models.BPE(unk_token='<unk>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.train_from_iterator(bpe_texts(), trainers.BpeTrainer(
    vocab_size=CFG['vocab'], min_frequency=2, special_tokens=['<pad>', '<eos>', '<unk>'],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
tok.save(str(OUT / 'tokenizer.json'))
CFG.update(vocab=tok.get_vocab_size(), pad=tok.token_to_id('<pad>'),
           eos=tok.token_to_id('<eos>'), unk=tok.token_to_id('<unk>'))
print(f'BPE vocabulary: {CFG["vocab"]:,}')

BPE vocabulary: 32,768


In [7]:
dev, counts = [], {}
for source, weight in zip(SOURCES, MIX):
    totals, selected, pending = [0, 0], [], []
    limits = [int(400000000 * weight), 500000]
    with (DATA / f'{source}.train.bin').open('wb') as train, (DATA / f'{source}.val.bin').open('wb') as val:
        files = [train, val]
        for text, heldout, key in documents(source):
            split = int(heldout)
            if heldout and key % 1000 >= 8:
                group = 'tune' if key % 1000 == 8 else 'audit'
                words = text.split()
                if sum(r['split'] == group for r in selected) < 8 and len(words) >= 640:
                    start = (key // 1000) % (len(words) - 639)
                    selected.append(dict(source=source, split=group, prompt=' '.join(words[start:start+512]),
                                         reference=' '.join(words[start+512:start+640])))
                continue
            if totals[split] < limits[split]:
                pending.append((text, split))
            if len(pending) >= 32:
                encoded = tok.encode_batch([x[0] for x in pending])
                for enc, (_, part) in zip(encoded, pending):
                    ids = np.asarray(enc.ids + [CFG['eos']], dtype=np.uint16)
                    ids = ids[:max(0, limits[part] - totals[part])]
                    files[part].write(ids.tobytes())
                    totals[part] += len(ids)
                pending.clear()
            if all(n >= lim for n, lim in zip(totals, limits)):
                break
        for enc, (_, part) in zip(tok.encode_batch([x[0] for x in pending]), pending):
            ids = np.asarray(enc.ids + [CFG['eos']], dtype=np.uint16)
            ids = ids[:max(0, limits[part] - totals[part])]
            files[part].write(ids.tobytes())
            totals[part] += len(ids)
    dev.extend(selected)
    counts[source] = dict(train=totals[0], val=totals[1])
    print(f'{source}: {totals[0]:,} train / {totals[1]:,} validation tokens')
(OUT / 'dev.json').write_text(json.dumps(dev))
(OUT / 'data_counts.json').write_text(json.dumps(counts, indent=2))
(OUT / 'config.json').write_text(json.dumps(CFG, indent=2))

acts: 40,000,000 train / 500,000 validation tokens
high_courts: 160,000,000 train / 500,000 validation tokens
supreme_court: 140,000,000 train / 500,000 validation tokens
tribunals: 60,000,000 train / 500,000 validation tokens


645

## 2. LegalBlock-GQA architecture

In [8]:
class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight

class Attention(nn.Module):
    def __init__(self, c, global_attn):
        super().__init__()
        self.h, self.kv, self.d = c['heads'], c['kv_heads'], c['dim'] // c['heads']
        self.window, self.global_attn = c['window'], global_attn
        self.qkv = nn.Linear(c['dim'], (self.h + 2*self.kv)*self.d, bias=False)
        self.out = nn.Linear(c['dim'], c['dim'], bias=False)
        self.register_buffer('freq', 1 / (10000 ** (torch.arange(0, self.d, 2).float()/self.d)), persistent=False)
    def rope(self, x, pos):
        angle = (torch.arange(pos, pos+x.size(2), device=x.device).float()[:, None] * self.freq).to(x.dtype)
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), -1).flatten(-2)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        b, t, _ = x.shape
        q, k, v = self.qkv(x).split([self.h*self.d, self.kv*self.d, self.kv*self.d], -1)
        q = self.rope(q.view(b,t,self.h,self.d).transpose(1,2), pos)
        k = self.rope(k.view(b,t,self.kv,self.d).transpose(1,2), pos)
        v = v.view(b,t,self.kv,self.d).transpose(1,2)
        if cache is not None and (self.global_attn or pos % self.window != 0):
            k, v = torch.cat((cache[0], k), 2), torch.cat((cache[1], v), 2)
        saved = None
        if use_cache:
            start = 0 if self.global_attn else ((pos+t-1)//self.window)*self.window - pos
            start = max(0, start) if cache is None else 0
            saved = (k[:,:,start:], v[:,:,start:])
        k, v = k.repeat_interleave(self.h//self.kv, 1), v.repeat_interleave(self.h//self.kv, 1)
        if t == 1:
            y = F.scaled_dot_product_attention(q, k, v)
        elif self.global_attn:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            w = self.window
            pad = (-t) % w
            q, k, v = [F.pad(z, (0,0,0,pad)).reshape(b,self.h,-1,w,self.d)
                       .permute(0,2,1,3,4).reshape(-1,self.h,w,self.d) for z in (q,k,v)]
            y = F.scaled_dot_product_attention(q,k,v,is_causal=True)
            y = y.reshape(b,-1,self.h,w,self.d).permute(0,2,1,3,4).reshape(b,self.h,-1,self.d)[:,:,:t]
        return self.out(y.transpose(1,2).reshape(b,t,-1)), saved

class Block(nn.Module):
    def __init__(self, c, i):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c['dim']), RMSNorm(c['dim'])
        self.attn = Attention(c, (i+1) % c['global_every'] == 0)
        self.up = nn.Linear(c['dim'], 2*c['ff'], bias=False)
        self.down = nn.Linear(c['ff'], c['dim'], bias=False)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        y, cache = self.attn(self.n1(x), cache, pos, use_cache)
        x = x + y
        a, b = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(a)*b), cache

class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(c['vocab'], c['dim'])
        self.blocks = nn.ModuleList([Block(c, i) for i in range(c['layers'])])
        self.norm = RMSNorm(c['dim'])
        self.apply(self.init_weights)
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=.02/math.sqrt(2*c['layers']))
            nn.init.normal_(block.down.weight, std=.02/math.sqrt(2*c['layers']))
    def init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=.02)
    def forward(self, ids, targets=None, cache=None, pos=0, use_cache=False):
        x, saved = self.emb(ids), []
        for i, block in enumerate(self.blocks):
            if self.training:
                x, _ = checkpoint(block, x, use_reentrant=False)
            else:
                x, kv = block(x, None if cache is None else cache[i], pos, use_cache)
                saved.append(kv)
        x = self.norm(x)
        if targets is None:
            return F.linear(x[:,-1], self.emb.weight), saved
        logits = F.linear(x, self.emb.weight)
        return F.cross_entropy(logits.flatten(0,1), targets.flatten(), ignore_index=-100)

## 3. DDP training and checkpoints

In [9]:
def batch(arrays, rng, c, length, device):
    sources = rng.choice(len(arrays), c['micro'], p=c['mix'])
    rows = [np.asarray(arrays[j][s:s+length+1], dtype=np.int64)
            for j in sources for s in [rng.integers(len(arrays[j])-length)]]
    z = torch.from_numpy(np.stack(rows)).pin_memory().to(device, non_blocking=True)
    x, y = z[:,:-1], z[:,1:].clone()
    y[x == c['eos']] = -100  # Do not predict the next document from the previous one.
    return x, y

def train_worker(rank, c, resume, session_seconds):
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    dist.init_process_group('nccl', rank=rank, world_size=2, timeout=timedelta(minutes=30))
    device, out = torch.device('cuda', rank), Path(c['out'])
    torch.manual_seed(c['seed'])
    model = LegalLM(c).to(device)
    decay = [p for p in model.parameters() if p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.ndim < 2]
    opt = torch.optim.AdamW([{'params':decay, 'weight_decay':c['wd']},
                            {'params':no_decay, 'weight_decay':0.}],
                           lr=c['lr'], betas=(.9,.95), eps=1e-8, fused=True)
    scaler = torch.amp.GradScaler('cuda', init_scale=4096)
    rng = np.random.default_rng(c['seed'] + rank)
    step, tokens, seconds, best = 0, 0, 0., float('inf')
    if resume:
        ck = torch.load(resume, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        step, tokens, seconds, best = ck['step'], ck['tokens'], ck['seconds'], ck['best']
        rng.bit_generator.state = ck['rng'][rank]['numpy']
        torch.set_rng_state(ck['rng'][rank]['torch'])
        torch.cuda.set_rng_state(ck['rng'][rank]['cuda'], device)
        del ck
    ddp = DDP(model, device_ids=[rank], broadcast_buffers=False, gradient_as_bucket_view=True)
    train = [np.memmap(Path(c['data'])/f'{s}.train.bin', dtype=np.uint16, mode='r') for s in c['sources']]
    val = [np.memmap(Path(c['data'])/f'{s}.val.bin', dtype=np.uint16, mode='r') for s in c['sources']]
    if rank == 0:
        print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters | resume step: {step:,}', flush=True)
    start, prior, last_log, log_tokens = time.monotonic(), seconds, time.monotonic(), tokens
    total_loss = torch.zeros((), device=device)
    log_steps = 0
    while True:
        control = torch.tensor([prior+time.monotonic()-start, time.monotonic()-start], device=device)
        dist.broadcast(control, 0)
        seconds, session_elapsed = control.tolist()
        progress = max(tokens/c['train_tokens'], seconds/c['train_seconds'])
        if progress >= 1 or session_elapsed >= session_seconds:
            break
        length = c['long_seq'] if progress >= .75 else c['seq']
        lr = c['min_lr'] + .5*(c['lr']-c['min_lr'])*(1+math.cos(math.pi*min(1,progress)))
        lr *= min(1., (step+1)/c['warmup'])
        for group in opt.param_groups:
            group['lr'] = lr
        ddp.train()
        opt.zero_grad(set_to_none=True)
        loss_sum = torch.zeros((), device=device)
        for micro in range(c['accum']):
            x, y = batch(train, rng, c, length, device)
            with ddp.no_sync() if micro < c['accum']-1 else nullcontext():
                with torch.autocast('cuda', dtype=torch.float16):
                    loss = ddp(x,y) / c['accum']
                scaler.scale(loss).backward()
            loss_sum += loss.detach()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        scaler.step(opt)
        scaler.update()
        step += 1
        tokens += length*c['micro']*2*c['accum']
        total_loss += loss_sum
        log_steps += 1
        if step % 25 == 0:
            dist.all_reduce(total_loss)
            if rank == 0:
                rate = (tokens-log_tokens)/(time.monotonic()-last_log)
                print(f'Step {step:,} | {tokens/1e6:.1f}M tokens | L={length} | loss {total_loss.item()/(2*log_steps):.4f} | {rate:,.0f} tok/s', flush=True)
            total_loss.zero_()
            log_steps, last_log, log_tokens = 0, time.monotonic(), tokens
        if step % c['eval_every'] == 0:
            best = evaluate_save(model, opt, scaler, val, rng, rank, c, step, tokens,
                                 prior+time.monotonic()-start, best)
    evaluate_save(model, opt, scaler, val, rng, rank, c, step, tokens,
                  prior+time.monotonic()-start, best)
    dist.destroy_process_group()

def evaluate_save(model, opt, scaler, val, rng, rank, c, step, tokens, seconds, best):
    model.eval()
    device, out = next(model.parameters()).device, Path(c['out'])
    vrng = np.random.default_rng(9000+rank)
    sums = torch.zeros(2, device=device)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for _ in range(c['val_batches']):
            x,y = batch(val, vrng, c, c['long_seq'], device)
            n = (y != -100).sum()
            sums[0] += model(x,y)*n
            sums[1] += n
    dist.all_reduce(sums)
    value = (sums[0]/sums[1]).item()
    states = [None, None]
    dist.all_gather_object(states, dict(numpy=rng.bit_generator.state, torch=torch.get_rng_state(),
                                      cuda=torch.cuda.get_rng_state(device)))
    if rank == 0:
        if value < best:
            best = value
            torch.save(dict(model={k:v.detach().cpu().half() for k,v in model.state_dict().items()},
                            config=c, step=step, val_loss=value), out/'best.tmp')
            os.replace(out/'best.tmp', out/'best.pt')
        torch.save(dict(model=model.state_dict(), optimizer=opt.state_dict(), scaler=scaler.state_dict(),
                        config=c, step=step, tokens=tokens, seconds=seconds, best=best, rng=states), out/'last.tmp')
        os.replace(out/'last.tmp', out/'last.pt')
        with (out/'losses.jsonl').open('a') as f:
            f.write(json.dumps(dict(step=step, tokens=tokens, seconds=seconds, val_loss=value))+'\n')
        print(f'Validation {value:.4f} | best {best:.4f} | saved step {step:,}', flush=True)
    best_tensor = torch.tensor(best, device=device)
    dist.broadcast(best_tensor, 0)
    dist.barrier()
    return best_tensor.item()

In [10]:
def launch(fn, *args):
    payload = OUT / 'worker.pkl'
    with payload.open('wb') as f:
        cloudpickle.dump((fn,args), f)
    with socket.socket() as sock:
        sock.bind(('127.0.0.1',0))
        port = sock.getsockname()[1]
    command = "import os,cloudpickle; f=open(os.environ['WORKER_PAYLOAD'],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(int(os.environ['RANK']),*args)"
    workers = [subprocess.Popen([sys.executable,'-u','-c',command], env=dict(os.environ,
               RANK=str(rank), WORLD_SIZE='2', LOCAL_RANK=str(rank), MASTER_ADDR='127.0.0.1',
               MASTER_PORT=str(port), WORKER_PAYLOAD=str(payload))) for rank in range(2)]
    for p in workers:
        p.wait()
    payload.unlink()
    for p in workers:
        if p.returncode:
            raise subprocess.CalledProcessError(p.returncode, p.args)

CFG = json.loads((OUT/'config.json').read_text())
CFG.update(out=str(OUT), data=str(DATA))
remaining = max(900, 11*3600 - (time.monotonic()-SESSION_START) - 3600)
launch(train_worker, CFG, RESUME, remaining)

Model: 261,535,680 parameters | resume step: 5,722
Validation 2.0385 | best 2.0385 | saved step 5,722


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


## 4. Generation and selection

In [11]:
@torch.inference_mode()
def generate(model, tok, text, setting, seed):
    device = next(model.parameters()).device
    torch.manual_seed(seed)
    ids = tok.encode(text).ids
    x = torch.tensor([ids], device=device)
    result, cache, pos = [], None, 0
    words = []
    with torch.autocast('cuda', dtype=torch.float16):
        for i in range(768):
            logits, cache = model(x, cache=cache, pos=pos, use_cache=True)
            pos += x.size(1)
            logits = logits[0].float()
            logits[[model.c['pad'],model.c['eos'],model.c['unk']]] = -float('inf')
            if result:
                repeated = torch.tensor(list(set(result[-96:])), device=device)
                z = logits[repeated]
                logits[repeated] = torch.where(z < 0, z*1.08, z/1.08)
            if setting == 'greedy':
                nxt = logits.argmax()
            else:
                scores, indices = (logits/.7).topk(40)
                probs = scores.softmax(-1)
                scores[(probs.cumsum(-1)-probs) > .9] = -float('inf')
                nxt = indices[torch.multinomial(scores.softmax(-1),1)[0]]
            result.append(nxt.item())
            x = nxt.reshape(1,1)
            if (i+1) % 8 == 0:
                words = tok.decode(result).split()
                if len(words) >= 129:
                    break
    return ' '.join(tok.decode(result).split()[:128])

def inference_worker(rank, c, jobs, destination):
    torch.set_num_threads(2)
    torch.cuda.set_device(rank)
    tok = Tokenizer.from_file(str(Path(c['out'])/'tokenizer.json'))
    current, model, answers = None, None, []
    for j, job in enumerate(jobs[rank::2]):
        if job['checkpoint'] != current:
            current = job['checkpoint']
            del model
            torch.cuda.empty_cache()
            ck = torch.load(current, map_location='cpu', weights_only=False)
            model = LegalLM(ck['config'])
            model.load_state_dict(ck['model'])
            model = model.to(device=f'cuda:{rank}', dtype=torch.float16).eval()
            del ck
        text = generate(model, tok, job['prompt'], job['setting'], job['seed'])
        answers.append(dict(id=job['id'], checkpoint=job['checkpoint'], setting=job['setting'], text=text))
        if rank == 0 and (j+1) % 25 == 0:
            print(f'Generated {j+1}/{len(jobs[rank::2])} on GPU 0', flush=True)
    Path(f'{destination}.{rank}.json').write_text(json.dumps(answers))

In [12]:
dev = json.loads((OUT/'dev.json').read_text())
tune = [r for r in dev if r['split']=='tune']
audit = [r for r in dev if r['split']=='audit']
print(f'Continuation split: {len(tune)} tuning / {len(audit)} audit examples')
checkpoints = [str(OUT/'best.pt'), str(OUT/'last.pt')]
settings = ['greedy','sample']
jobs = [dict(id=str(i), prompt=row['prompt'], checkpoint=ck, setting=setting, seed=781+i)
        for ck in checkpoints for setting in settings for i,row in enumerate(tune)]
launch(inference_worker, CFG, jobs, str(OUT/'tune'))
answers = sum([json.loads(Path(f'{OUT}/tune.{r}.json').read_text()) for r in range(2)], [])

from bert_score import BERTScorer
scorer = BERTScorer(model_type='roberta-large', num_layers=17, lang='en',
                    device='cuda:0', batch_size=8, rescale_with_baseline=False)
results = []
for ck in checkpoints:
    for setting in settings:
        rows = sorted([a for a in answers if a['checkpoint']==ck and a['setting']==setting], key=lambda a:int(a['id']))
        _,_,f1 = scorer.score([r['text'] for r in rows], [r['reference'] for r in tune])
        score = f1.mean().item()
        results.append(dict(checkpoint=ck, setting=setting, bert_f1=score))
        print(f'{Path(ck).name} / {setting}: BERTScore F1 {score:.4f}')
choice = max(results, key=lambda r:r['bert_f1'])
(OUT/'selection.json').write_text(json.dumps(dict(choice=choice, results=results,
    metric='roberta-large L17, no IDF, no baseline rescaling'), indent=2))
del scorer
gc.collect()
torch.cuda.empty_cache()
print(f'Selected: {Path(choice["checkpoint"]).name} / {choice["setting"]}')

Continuation split: 32 tuning / 32 audit examples
Generated 25/64 on GPU 0
Generated 50/64 on GPU 0


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


best.pt / greedy: BERTScore F1 0.8507
best.pt / sample: BERTScore F1 0.8597
last.pt / greedy: BERTScore F1 0.8507
last.pt / sample: BERTScore F1 0.8597
Selected: best.pt / sample


In [13]:
choice = json.loads((OUT/'selection.json').read_text())['choice']
jobs = [dict(id=str(i), prompt=row['prompt'], checkpoint=choice['checkpoint'], setting=choice['setting'], seed=1781+i)
        for i,row in enumerate(audit)]
launch(inference_worker, CFG, jobs, str(OUT/'audit'))
audit_answers = sum([json.loads(Path(f'{OUT}/audit.{r}.json').read_text()) for r in range(2)], [])
audit_answers.sort(key=lambda a:int(a['id']))
scorer = BERTScorer(model_type='roberta-large', num_layers=17, lang='en',
                    device='cuda:0', batch_size=8, rescale_with_baseline=False)
pred, refs = [x['text'] for x in audit_answers], [x['reference'] for x in audit]
_,_,f1 = scorer.score(pred, refs)
from rouge_score import rouge_scorer
from sacrebleu.metrics import BLEU
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
metrics = dict(bert_f1=f1.mean().item(), rougeL=float(np.mean([rouge.score(r,p)['rougeL'].fmeasure for p,r in zip(pred,refs)])),
               bleu4=BLEU().corpus_score(pred,[refs]).score)
(OUT/'audit_metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'Held-out audit: BERTScore {metrics["bert_f1"]:.4f} | ROUGE-L {metrics["rougeL"]:.4f} | BLEU-4 {metrics["bleu4"]:.2f}')
del scorer
gc.collect()
torch.cuda.empty_cache()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Held-out audit: BERTScore 0.8565 | ROUGE-L 0.3033 | BLEU-4 20.41


## 5. Official predictions

In [14]:
test_path = hf_hub_download(REPO, 'Phase_1_Pretraining/Test/test.json', repo_type='dataset')
test = json.loads(Path(test_path).read_text())
jobs = [dict(id=key, prompt=row['text'], checkpoint=choice['checkpoint'], setting=choice['setting'], seed=2781+i)
        for i,(key,row) in enumerate(test.items())]
launch(inference_worker, CFG, jobs, str(OUT/'test'))
rows = sum([json.loads(Path(f'{OUT}/test.{r}.json').read_text()) for r in range(2)], [])
by_id = {row['id']:row['text'] for row in rows}
prediction = {key:{'text':by_id[key]} for key in test}
(OUT/'prediction.json').write_text(json.dumps(prediction, ensure_ascii=False))
with zipfile.ZipFile(OUT/'submission.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT/'prediction.json', arcname='prediction.json')
short = sum(len(row['text'].split()) < 128 for row in prediction.values())
print(f'{len(prediction)} predictions | {short} shorter than 128 words | {OUT / "submission.zip"}')

test.json: 0.00B [00:00, ?B/s]

Generated 25/500 on GPU 0
Generated 50/500 on GPU 0
Generated 75/500 on GPU 0
Generated 100/500 on GPU 0
Generated 125/500 on GPU 0
Generated 150/500 on GPU 0
Generated 175/500 on GPU 0
Generated 200/500 on GPU 0
Generated 225/500 on GPU 0
Generated 250/500 on GPU 0
Generated 275/500 on GPU 0
Generated 300/500 on GPU 0
Generated 325/500 on GPU 0
Generated 350/500 on GPU 0
Generated 375/500 on GPU 0
Generated 400/500 on GPU 0
Generated 425/500 on GPU 0
Generated 450/500 on GPU 0
Generated 475/500 on GPU 0
Generated 500/500 on GPU 0
1000 predictions | 1 shorter than 128 words | /kaggle/working/legalblock_gqa/submission.zip


In [15]:
bundle = OUT / "legalblock_gqa_output_bundle.zip"

files = [
    OUT / "best.pt",
    OUT / "last.pt",
    OUT / "tokenizer.json",
    OUT / "config.json",
    OUT / "data_counts.json",
    OUT / "dev.json",
    OUT / "selection.json",
    OUT / "audit_metrics.json",
    OUT / "losses.jsonl",
    OUT / "prediction.json",
    OUT / "submission.zip",
]
files += sorted(DATA.glob("*.bin"))
files += sorted(OUT.glob("tune.*.json"))
files += sorted(OUT.glob("audit.*.json"))
files += sorted(OUT.glob("test.*.json"))

with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_STORED) as z:
    for path in files:
        if path.is_file():
            z.write(path, arcname=str(Path("legalblock_gqa") / path.relative_to(OUT)))

print(f"Bundle: {bundle}")
print(f"Included {sum(path.is_file() for path in files)} files | {bundle.stat().st_size / 1e9:.2f} GB")

Bundle: /kaggle/working/legalblock_gqa/legalblock_gqa_output_bundle.zip
Included 25 files | 4.47 GB


## Architecture summary